In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Mundka, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,198.55,349.19,25.96,37.07,40.83,32.31,7.58,1.13,18.40,1.60,26.48,86.84,0.73,63.15,979.57,12.12,0.0,0
1,02-01-2025 00:00,03-01-2025 00:00,195.00,349.71,12.84,36.76,29.99,44.34,7.96,1.32,17.48,1.65,19.65,88.19,0.70,84.09,979.58,12.32,0.0,0
2,03-01-2025 00:00,04-01-2025 00:00,212.33,363.52,40.41,42.21,54.95,61.34,8.76,1.87,14.07,2.49,54.60,89.49,0.80,97.65,979.68,13.12,0.0,0
3,04-01-2025 00:00,05-01-2025 00:00,302.58,455.04,23.79,74.47,58.95,69.95,7.84,2.28,20.96,3.61,67.95,86.58,0.71,119.04,979.68,14.07,0.0,0
4,05-01-2025 00:00,06-01-2025 00:00,208.42,388.21,13.49,50.77,37.97,52.12,4.69,1.42,17.82,1.87,21.56,82.64,0.71,126.95,979.55,13.85,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,362.75,438.92,42.46,100.79,87.39,58.63,62.05,1.59,35.50,2.72,68.21,61.42,0.84,110.62,974.71,19.16,0.0,0
316,13-11-2025 00:00,14-11-2025 00:00,327.54,393.75,41.57,103.91,89.07,57.42,25.61,1.07,31.43,1.78,103.79,64.02,0.72,118.12,975.18,19.01,0.0,0
317,14-11-2025 00:00,15-11-2025 00:00,284.33,361.50,36.59,104.62,85.40,52.96,25.16,1.39,30.89,1.87,100.82,62.25,0.72,115.76,975.25,18.79,0.0,0
318,15-11-2025 00:00,16-11-2025 00:00,280.42,347.96,41.18,97.85,85.53,53.07,28.31,1.45,32.41,1.68,80.23,60.91,0.80,111.66,975.77,18.79,0.0,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date    PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  198.550  349.19  25.96  37.07  40.83   
1  02-01-2025 00:00  03-01-2025 00:00  195.000  349.71  12.84  36.76  29.99   
2  03-01-2025 00:00  04-01-2025 00:00  212.330  363.52  40.41  42.21  54.95   
3  04-01-2025 00:00  05-01-2025 00:00   69.145  455.04  23.79  74.47  58.95   
4  05-01-2025 00:00  06-01-2025 00:00  208.420  388.21  13.49  50.77  37.97   

     NH3   SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD      BP  \
0  32.31  7.58  1.13  18.40     1.60    26.48  86.84  0.73   63.15  979.57   
1  44.34  7.96  1.32  17.48     1.65    19.65  88.19  0.70   84.09  979.58   
2  61.34  8.76  0.67  14.07     2.49    54.60  89.49  0.80   97.65  979.68   
3  29.59  7.84  0.67  20.96     3.61    67.95  86.58  0.71  119.04  979.68   
4  52.12  4.69  0.67  17.82     1.87    21.56  82.64  0.71  126.95  979.55   

      AT   RF  TOT-RF  
0  12.12 

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,1.951180,0.860704,1.332072,-0.207038,0.594945,0.133056,-0.443440,1.957357,-1.269502,0.156301,-0.666655,1.734525,-1.030338,-2.159420,1.573086,-2.316032,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,1.891215,0.865308,-0.007400,-0.227972,-0.130319,0.959578,-0.334622,2.805955,-1.344515,0.219548,-1.043520,1.837662,-1.086236,-1.028894,1.578627,-2.285490,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,2.183948,0.987563,2.807329,0.140076,1.539662,2.127563,-0.105533,-0.097142,-1.622549,1.282109,0.884946,1.936979,-0.899908,-0.296805,1.634037,-2.163322,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.234690,1.797760,1.110528,2.318651,1.807287,-0.053822,-0.368986,-0.097142,-1.060773,2.698856,1.621570,1.714662,-1.067604,0.858016,1.634037,-2.018248,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.117901,1.206136,0.058961,0.718148,0.403593,1.494103,-1.271024,-0.097142,-1.316793,0.497838,-0.938130,1.413656,-1.067604,1.285068,1.562004,-2.051844,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.234690,1.655055,3.016621,-0.173609,-0.201909,1.941373,-0.161374,-0.097142,0.124745,1.573048,1.635916,-0.207501,-0.825377,0.403430,-1.119833,-1.240956,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.234690,1.255179,2.925758,-0.173609,-0.201909,1.858240,-0.161374,1.689379,-0.207103,0.383992,-0.102738,-0.008867,-1.048971,0.808346,-0.859407,-1.263863,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.234690,0.969681,2.417330,-0.173609,-0.201909,1.551815,-0.161374,-0.097142,-0.251131,0.497838,-0.102738,-0.144091,-1.048971,0.680933,-0.820620,-1.297459,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.234690,0.849816,2.885941,-0.173609,-0.201909,1.559373,-0.161374,-0.097142,-0.127198,0.257497,2.299154,-0.246463,-0.899908,0.459578,-0.532489,-1.297459,0.0,0.0


In [10]:
df.to_excel('mundka2025.xlsx', index=False)